In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
DATASET_PATH = r"/content/drive/MyDrive/Colab Notebooks/SProject/train_processed.csv"
LABEL_COL = "Label"

df = pd.read_csv(DATASET_PATH)

X = df.drop(columns=[LABEL_COL]).values.astype(np.float32)  # ép float32
y = df[LABEL_COL].values

print(X.shape, y.shape)


(2264594, 64) (2264594,)


In [4]:
le = LabelEncoder()
y_enc = le.fit_transform(y)

num_classes = len(np.unique(y_enc))

In [5]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y_enc,
    test_size=0.2,
    random_state=42,
    stratify=y_enc
)


In [6]:
def sequence_generator(X, y, timesteps, batch_size):
    num_samples = len(X) - timesteps

    while True:
        for start in range(0, num_samples, batch_size):
            end = min(start + batch_size, num_samples)

            X_batch = []
            y_batch = []

            for i in range(start, end):
                X_batch.append(X[i:i+timesteps])
                y_batch.append(y[i+timesteps])

            yield np.array(X_batch), tf.keras.utils.to_categorical(
                np.array(y_batch), num_classes
            )


In [7]:
TIMESTEPS = 10
BATCH_SIZE = 256

model = Sequential([
    LSTM(64, input_shape=(TIMESTEPS, X.shape[1])),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 15)             │           975 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 38,159 (149.06 KB)

 Trainable params: 38,159 (149.06 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
steps_per_epoch = (len(X_train) - TIMESTEPS) // BATCH_SIZE
val_steps = (len(X_val) - TIMESTEPS) // BATCH_SIZE

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    sequence_generator(X_train, y_train, TIMESTEPS, BATCH_SIZE),
    steps_per_epoch=steps_per_epoch,
    validation_data=sequence_generator(X_val, y_val, TIMESTEPS, BATCH_SIZE),
    validation_steps=val_steps,
    epochs=20,
    callbacks=[early_stop],
    verbose=1
)


Epoch 1/20
7076/7076 ━━━━━━━━━━━━━━━━━━━━ 78s 10ms/step - accuracy: 0.7980 - loss: 0.8199 - val_accuracy: 0.8030 - val_loss: 0.7686
Epoch 2/20
7076/7076 ━━━━━━━━━━━━━━━━━━━━ 74s 11ms/step - accuracy: 0.8028 - loss: 0.7716 - val_accuracy: 0.8030 - val_loss: 0.7686
Epoch 3/20
7076/7076 ━━━━━━━━━━━━━━━━━━━━ 73s 10ms/step - accuracy: 0.8028 - loss: 0.7705 - val_accuracy: 0.8030 - val_loss: 0.7688
Epoch 4/20
7076/7076 ━━━━━━━━━━━━━━━━━━━━ 74s 10ms/step - accuracy: 0.8028 - loss: 0.7701 - val_accuracy: 0.8030 - val_loss: 0.7686
Epoch 5/20
7076/7076 ━━━━━━━━━━━━━━━━━━━━ 71s 10ms/step - accuracy: 0.8028 - loss: 0.7698 - val_accuracy: 0.8030 - val_loss: 0.7687
Epoch 6/20
7076/7076 ━━━━━━━━━━━━━━━━━━━━ 74s 10ms/step - accuracy: 0.8029 - loss: 0.7696 - val_accuracy: 0.8030 - val_loss: 0.7686
Epoch 7/20
7076/7076 ━━━━━━━━━━━━━━━━━━━━ 75s 11ms/step - accuracy: 0.8028 - loss: 0.7697 - val_accuracy: 0.8030 - val_loss: 0.7686


In [11]:
'''
Save the trained model
'''
model.save(r"/content/drive/MyDrive/Colab Notebooks/SProject/models/model.keras")

print(f"Model saved")

Model saved
